In [3]:
import json

import pandas as pd
import requests

# Exact endpoint + apiKey you provided
url = "https://www.krl.dk/sirka/sirkaApi/tableApi"
api_key = "b6caa39c5a6dc520a83ed1624a7bdfdd1935d9fb2bee1d5e2c14dc08ee4ea57ceacca8e9550f5b25ad07b745a612d4b6b5b07bd781baff0908b5891c0beb65c1"

# Exact payload from KRL (only wrapped into a Python dict and with apiKey inserted)
payload = {
    "apiKey": api_key,
    "table": "Personale-måned",
    "time": [
        {"y1": "2026", "m1": "01"},
        {"y1": "2025", "m1": "01"},
        {"y1": "2024", "m1": "01"},
        {"y1": "2023", "m1": "01"},
        {"y1": "2022", "m1": "01"},
        {"y1": "2021", "m1": "01"},
        {"y1": "2020", "m1": "01"},
        {"y1": "2019", "m1": "01"},
        {"y1": "2018", "m1": "01"},
    ],
    "control": ["kom_reg"],
    "data": ["fuldtid"],
    "selection": [
        {
            "name": "Udvalgte population",
            "filters": {"omr": ["1", "8"]},
        }
    ],
    "options": {
        "totals": True,
        "actions": [],
        "tableName": "Antal ansatte",
        "subLimit": 5,
        "modelName": "SIRKA",
        "timeIncreasing": True,
        # Ask API to return an Excel file directly
        "outputFormat": "excel",
    },
    "dimension": {
        "viewportHeight": 812,
        "viewportWidth": 1440,
        "xsMaxWidth": 768,
        "smMaxWidth": 992,
        "mdMaxWidth": 1200,
        "CONSTANTS": {"XS": 0, "SM": 1, "MD": 2, "LG": 3, "MAIL": 4},
    },
}

# Call API
response = requests.post(url, json=payload, timeout=60)
print("HTTP:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))

out_path = "Figur1_api_output.xlsx"

# If API returns an Excel file, save it directly (no JSON->Excel conversion)
content_type = (response.headers.get("content-type") or "").lower()
if "spreadsheet" in content_type or "ms-excel" in content_type or out_path.lower().endswith(".xlsx") and response.content[:2] == b"PK":
    with open(out_path, "wb") as f:
        f.write(response.content)
    print("Wrote (direct from API):", out_path)
else:
    # Fallback: API returned JSON; save it to Excel without calculations
    api_json = response.json()
    try:
        import openpyxl  # noqa: F401
    except ImportError as e:
        raise ImportError("Missing dependency: openpyxl. Install with: pip install openpyxl") from e
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        if isinstance(api_json, list):
            pd.DataFrame(api_json).to_excel(writer, sheet_name="api", index=False)
            print("Rows:", len(api_json))
        else:
            pd.DataFrame({"json": [json.dumps(api_json, ensure_ascii=False)]}).to_excel(writer, sheet_name="api", index=False)
            print("Non-list JSON saved (dict/error).")
    print("Wrote (converted from JSON):", out_path)

HTTP: 200
Content-Type: application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
Wrote (direct from API): Figur1_api_output.xlsx
